In [2]:
# ============================================================
# DATA PREPROCESSING
#
# Project:
# Predictive Analysis of Schedule Delays and Cost Overruns
# in Building Construction Projects Using Machine Learning
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 2. LOAD RAW DATA
# ============================================================

file_path = "E:/Construction_Analysis_Project/Data/bim_ai_civil_engineering_dataset.csv"

df = pd.read_csv(file_path)

print("=" * 70)
print("RAW DATA")
print("=" * 70)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nOriginal columns:")
print(df.columns.tolist())


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nCleaned column names:")
print(df.columns.tolist())


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [

    # Identification
    "Project_ID",
    "Project_Type",
    "Location",

    # Dates
    "Start_Date",
    "End_Date",

    # Schedule
    "Planned_Duration",
    "Actual_Duration",

    # Cost
    "Planned_Cost",
    "Actual_Cost",

    # Construction/project factors
    "Weather_Condition",
    "Temperature",
    "Humidity",
    "Air_Quality_Index",
    "Energy_Consumption",
    "Material_Usage",
    "Labor_Hours",
    "Equipment_Utilization",
    "Accident_Count",
    "Safety_Risk_Score",

    # Building/site condition
    "Vibration_Level",
    "Crack_Width",
    "Load_Bearing_Capacity"
]


missing_columns = [

    column
    for column in required_columns
    if column not in df.columns

]


if missing_columns:

    raise ValueError(
        "\nThe following required columns are missing:\n"
        + "\n".join(missing_columns)
    )


# ============================================================
# 5. KEEP ONLY RELEVANT COLUMNS
# ============================================================



df = df[required_columns].copy()


print("\nColumns retained for this project:")
print(df.columns.tolist())


# ============================================================
# 6. REMOVE DUPLICATE PROJECTS
# ============================================================

print("\nDuplicate Project IDs:")

print(
    df["Project_ID"].duplicated().sum()
)


df = df.drop_duplicates(

    subset="Project_ID",

    keep="first"

).copy()


# ============================================================
# 7. FILTER ONLY BUILDING PROJECTS
# ============================================================

df["Project_Type"] = (

    df["Project_Type"]
    .astype(str)
    .str.strip()
    .str.title()

)


print("\nProject types:")
print(
    df["Project_Type"].value_counts()
)


df = df[
    df["Project_Type"] == "Building"
].copy()


print(
    "\nBuilding projects remaining:",
    len(df)
)


# ============================================================
# 8. REMOVE PROJECT TYPE
# ============================================================

# After filtering, every project is Building.
# Therefore Project_Type carries no useful information.

df = df.drop(
    columns=["Project_Type"]
)


# ============================================================
# 9. CONVERT DATE VARIABLES
# ============================================================

df["Start_Date"] = pd.to_datetime(
    df["Start_Date"],
    errors="coerce"
)

df["End_Date"] = pd.to_datetime(
    df["End_Date"],
    errors="coerce"
)


# ============================================================
# 10. CREATE USEFUL DATE FEATURES
# ============================================================

df["Start_Year"] = (
    df["Start_Date"].dt.year
)

df["Start_Month"] = (
    df["Start_Date"].dt.month
)

df["Start_Quarter"] = (
    df["Start_Date"].dt.quarter
)


# Raw dates are no longer required for analysis.

df = df.drop(
    columns=[
        "Start_Date",
        "End_Date"
    ]
)


# ============================================================
# 11. CONVERT NUMERICAL VARIABLES
# ============================================================

numerical_columns = [

    "Planned_Duration",
    "Actual_Duration",

    "Planned_Cost",
    "Actual_Cost",

    "Temperature",
    "Humidity",
    "Air_Quality_Index",

    "Energy_Consumption",
    "Material_Usage",
    "Labor_Hours",
    "Equipment_Utilization",

    "Accident_Count",
    "Safety_Risk_Score",

    "Vibration_Level",
    "Crack_Width",
    "Load_Bearing_Capacity"
]


for column in numerical_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# ============================================================
# 12. CLEAN CATEGORICAL VARIABLES
# ============================================================

df["Location"] = (

    df["Location"]
    .astype(str)
    .str.strip()
    .str.title()

)


df["Weather_Condition"] = (

    df["Weather_Condition"]
    .astype(str)
    .str.strip()
    .str.title()

)


# Replaced strings created from missing values

df["Location"] = (
    df["Location"]
    .replace(["Nan", "None", ""], np.nan)
)


df["Weather_Condition"] = (
    df["Weather_Condition"]
    .replace(["Nan", "None", ""], np.nan)
)


# ============================================================
# 13. HANDLE IMPOSSIBLE VALUES
# ============================================================

# Cost cannot be zero or negative.

df.loc[
    df["Planned_Cost"] <= 0,
    "Planned_Cost"
] = np.nan


df.loc[
    df["Actual_Cost"] <= 0,
    "Actual_Cost"
] = np.nan


# Duration cannot be zero or negative.

df.loc[
    df["Planned_Duration"] <= 0,
    "Planned_Duration"
] = np.nan


df.loc[
    df["Actual_Duration"] <= 0,
    "Actual_Duration"
] = np.nan


# Variables that cannot logically be negative.

non_negative_columns = [

    "Temperature",
    "Humidity",
    "Air_Quality_Index",
    "Energy_Consumption",

    "Material_Usage",
    "Labor_Hours",
    "Equipment_Utilization",

    "Accident_Count",
    "Safety_Risk_Score",

    "Vibration_Level",
    "Crack_Width",
    "Load_Bearing_Capacity"
]


for column in non_negative_columns:

    df.loc[
        df[column] < 0,
        column
    ] = np.nan


# ============================================================
# 14. CREATE SCHEDULE DELAY %
# ============================================================

df["Delay_Percentage"] = (

    (
        df["Actual_Duration"]
        -
        df["Planned_Duration"]
    )

    /

    df["Planned_Duration"]

) * 100


# ============================================================
# 15. CREATE COST OVERRUN %
# ============================================================

df["Cost_Overrun_Percentage"] = (

    (
        df["Actual_Cost"]
        -
        df["Planned_Cost"]
    )

    /

    df["Planned_Cost"]

) * 100


# ============================================================
# 16. CREATE YES/NO SCHEDULE DELAY TARGET
# ============================================================

# Project is considered significantly delayed
# when delay is >= 10%.

DELAY_THRESHOLD = 10


df["Schedule_Delay"] = np.where(

    df["Delay_Percentage"] >= DELAY_THRESHOLD,

    "Yes",

    "No"

)


# ============================================================
# 17. CREATE YES/NO COST OVERRUN TARGET
# ============================================================

# Project is considered to have significant cost overrun
# when overrun is >= 10%.

COST_OVERRUN_THRESHOLD = 10


df["Cost_Overrun_Flag"] = np.where(

    df["Cost_Overrun_Percentage"]
    >=
    COST_OVERRUN_THRESHOLD,

    "Yes",

    "No"

)


# Renamed target to make it more readable.

df = df.rename(

    columns={
        "Cost_Overrun_Flag":
        "Cost_Overrun"
    }

)


# ============================================================
# 18. HANDLE MISSING VALUES
# ============================================================

# IMPORTANT:
#
# Actual Cost and Actual Duration are needed to calculate
# our target variables.
#
# Therefore, if these are missing, i cannot reliably
# determine whether the completed project was delayed/overrun.
#
# Such records are removed rather than inventing the outcome.

df = df.dropna(

    subset=[
        "Planned_Cost",
        "Actual_Cost",
        "Planned_Duration",
        "Actual_Duration"
    ]

).copy()


# ============================================================
# 19. IMPUTE MISSING NUMERICAL PREDICTORS
# ============================================================

predictor_numerical_columns = [

    "Temperature",
    "Humidity",
    "Air_Quality_Index",

    "Energy_Consumption",
    "Material_Usage",
    "Labor_Hours",
    "Equipment_Utilization",

    "Accident_Count",
    "Safety_Risk_Score",

    "Vibration_Level",
    "Crack_Width",
    "Load_Bearing_Capacity"
]


for column in predictor_numerical_columns:

    median_value = df[column].median()

    df[column] = df[column].fillna(
        median_value
    )


# ============================================================
# 20. IMPUTE MISSING CATEGORICAL VARIABLES
# ============================================================

df["Location"] = df["Location"].fillna(
    df["Location"].mode()[0]
)


df["Weather_Condition"] = (
    df["Weather_Condition"]
    .fillna(
        df["Weather_Condition"].mode()[0]
    )
)


# ============================================================
# 21. OUTLIER DETECTION USING IQR
# ============================================================

# detect outliers for EDA purposes.
#
# do NOT remove legitimate construction projects merely
# because they are unusually large or expensive.

outlier_summary = []


outlier_columns = [

    "Planned_Cost",
    "Actual_Cost",

    "Planned_Duration",
    "Actual_Duration",

    "Temperature",
    "Humidity",
    "Air_Quality_Index",

    "Energy_Consumption",
    "Material_Usage",
    "Labor_Hours",
    "Equipment_Utilization",

    "Accident_Count",
    "Safety_Risk_Score",

    "Vibration_Level",
    "Crack_Width",
    "Load_Bearing_Capacity"

]


for column in outlier_columns:

    Q1 = df[column].quantile(0.25)

    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = (
        Q1 - 1.5 * IQR
    )

    upper_bound = (
        Q3 + 1.5 * IQR
    )

    number_of_outliers = (

        (
            (df[column] < lower_bound)
            |
            (df[column] > upper_bound)
        )

        .sum()

    )


    outlier_summary.append({

        "Feature": column,

        "Q1": Q1,

        "Q3": Q3,

        "IQR": IQR,

        "Lower_Bound": lower_bound,

        "Upper_Bound": upper_bound,

        "Outlier_Count":
            number_of_outliers

    })


outlier_report = pd.DataFrame(
    outlier_summary
)


print("\nOutlier summary:")

print(
    outlier_report
)


# ============================================================
# 22. DO NOT SCALE OR ONE-HOT ENCODE HERE
# ============================================================

# IMPORTANT:
#
# This dataset is going to EDA first.
#
# Therefore:
#
# Location remains readable.
# Weather remains readable.
# Costs remain in their original units.
# Durations remain in days.
# Labor remains in hours.
#
# Encoding and scaling will be done later AFTER EDA,
# inside the ML pipeline.
#
# This keeps the EDA interpretable.


# ============================================================
# 23. REORDER COLUMNS
# ============================================================

final_columns = [

    # Identification
    "Project_ID",
    "Location",

    # Planning
    "Planned_Cost",
    "Planned_Duration",

    # Project timing
    "Start_Year",
    "Start_Month",
    "Start_Quarter",

    # Construction/environment
    "Weather_Condition",
    "Temperature",
    "Humidity",
    "Air_Quality_Index",
    "Energy_Consumption",

    # Resources
    "Material_Usage",
    "Labor_Hours",
    "Equipment_Utilization",

    # Safety/risk
    "Accident_Count",
    "Safety_Risk_Score",

    # Building/site condition
    "Vibration_Level",
    "Crack_Width",
    "Load_Bearing_Capacity",

    # Actual outcomes
    "Actual_Duration",
    "Actual_Cost",

    # Derived outcomes
    "Delay_Percentage",
    "Cost_Overrun_Percentage",

    # Classification targets
    "Schedule_Delay",
    "Cost_Overrun"

]


df = df[final_columns]


# ============================================================
# 24. FINAL DATA QUALITY CHECK
# ============================================================

print("\n")
print("=" * 70)
print("FINAL DATASET CHECK")
print("=" * 70)


print(
    "\nFinal shape:",
    df.shape
)


print(
    "\nRemaining missing values:"
)


print(
    df.isnull().sum()
)


print(
    "\nDuplicate rows:",
    df.duplicated().sum()
)


print(
    "\nSchedule Delay distribution:"
)


print(
    df["Schedule_Delay"].value_counts()
)


print(
    "\nCost Overrun distribution:"
)


print(
    df["Cost_Overrun"].value_counts()
)


# ============================================================
# 25. SAVE ONE FINAL PROCESSED DATASET
# ============================================================

output_file = (
    "building_final_processed_data.csv"
)


df.to_csv(

    output_file,

    index=False

)


# ============================================================
# 26. FINAL MESSAGE
# ============================================================

print("\n")
print("=" * 70)
print("DATA PREPROCESSING COMPLETED")
print("=" * 70)

print(
    f"\nFinal processed dataset saved as:\n{output_file}"
)

print(
    "\nThis is the ONLY dataset you need to take into EDA."
)

print(
    "\nNext step:"
)

print(
    "building_final_processed_data.csv"
    " → EDA"
)

RAW DATA
Rows: 1000
Columns: 28

Original columns:
['Project_ID', 'Project_Type', 'Location', 'Start_Date', 'End_Date', 'Planned_Cost', 'Actual_Cost', 'Cost_Overrun', 'Planned_Duration', 'Actual_Duration', 'Schedule_Deviation', 'Vibration_Level', 'Crack_Width', 'Load_Bearing_Capacity', 'Temperature', 'Humidity', 'Weather_Condition', 'Air_Quality_Index', 'Energy_Consumption', 'Material_Usage', 'Labor_Hours', 'Equipment_Utilization', 'Accident_Count', 'Safety_Risk_Score', 'Image_Analysis_Score', 'Anomaly_Detected', 'Completion_Percentage', 'Risk_Level']

Cleaned column names:
['Project_ID', 'Project_Type', 'Location', 'Start_Date', 'End_Date', 'Planned_Cost', 'Actual_Cost', 'Cost_Overrun', 'Planned_Duration', 'Actual_Duration', 'Schedule_Deviation', 'Vibration_Level', 'Crack_Width', 'Load_Bearing_Capacity', 'Temperature', 'Humidity', 'Weather_Condition', 'Air_Quality_Index', 'Energy_Consumption', 'Material_Usage', 'Labor_Hours', 'Equipment_Utilization', 'Accident_Count', 'Safety_Risk_Sco